# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/720-hz/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane: Freestyle — Growth / Recovery / Momentum Prediction.**

Why this one: the starter pipeline's own target (`is_declining_label = trend_direction == "down"`) is flagged in the lane guide as a beginner proxy — a bucket computed from the CURRENT 90-day window, not a future outcome. I want to spend the next 7 weeks building the honest version of the same problem: `prior feature window -> future target window`, with a real leakage audit instead of the shortcut. It's also the discipline I actually care about: I've already had to catch a system of mine (an AI agent) reporting a task as finished when it hadn't run — the fix wasn't a nicer output, it was verification I could trust. A prediction lane that only means something if the validation is airtight is that same muscle, applied to data instead of code.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"
if IN_COLAB:
  if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
  os.chdir(REPO_DIR)
else:
  while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df[["impressions_90d", "trend_pct", "trend_direction"]].head(3).to_string())
print("trend_pct range per trend_direction bucket:")
print(df.groupby("trend_direction")["trend_pct"].agg(["min", "max", "count"]))
print("trend_pct and impressions_90d both describe the SAME trailing 90-day window -- there is no separate future window here.")
print("trend_direction is a deterministic bucket of trend_pct (down <= -20%, stable -20..20%, up >= +20%), not an observed later outcome -- that gap is exactly what this lane is built to close.")


   impressions_90d  trend_pct trend_direction
0             3803      -41.4            down
1            15320      -57.7            down
2            12581      -60.9            down
trend_pct range per trend_direction bucket:
                   min      max  count
trend_direction                       
down            -100.0    -20.0  16262
flat               NaN      NaN      0
new                NaN      NaN      0
stable           -20.0     20.0   5962
up                20.0  44900.0   4388
trend_pct and impressions_90d both describe the SAME trailing 90-day window -- there is no separate future window here.
trend_direction is a deterministic bucket of trend_pct (down <= -20%, stable -20..20%, up >= +20%), not an observed later outcome -- that gap is exactly what this lane is built to close.


## 2. The question: decision, action, cost of a wrong call

**Decision:** which pages a content reviewer with fixed weekly capacity should look at first — prioritizing pages likely to keep declining while they still carry real demand, and separately flagging pages showing early recovery or momentum worth protecting.

**Who acts:** a content strategist / SEO editor who can realistically review a capped number of pages per cycle (this notebook uses 20 and 50 as illustrative capacities).

**Cost of a wrong call:**
- False positive (flagged as at-risk, wasn't): burns one of a reviewer's limited slots on a page that didn't need it — bounded, recoverable cost.
- False negative (a real decline missed): a page that still has demand keeps losing visibility unnoticed until the next cycle. The numbers below show most of today's declining pages aren't dead yet, so a miss is compounding, not neutral.

Because reviewer capacity is the scarce resource, this is a ranking problem where Precision@K matches the actual decision — not raw accuracy.

In [6]:
declining_with_demand = df[(df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)]
n = len(declining_with_demand)
print(f"Declining pages with real demand (impressions_90d >= 100): {n}")
for capacity in [20, 50]:
  weeks = n / capacity
  print(f"At {capacity} pages/week reviewer capacity: {weeks:.0f} weeks of backlog just to clear today's declining set.")
print("Reviewer capacity can't keep pace with the backlog -- which pages get reviewed first is the whole decision, not a detail.")

Declining pages with real demand (impressions_90d >= 100): 13152
At 20 pages/week reviewer capacity: 658 weeks of backlog just to clear today's declining set.
At 50 pages/week reviewer capacity: 263 weeks of backlog just to clear today's declining set.
Reviewer capacity can't keep pace with the backlog -- which pages get reviewed first is the whole decision, not a detail.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from `content_refresh_anonymized.csv` (computed in the cell below, not asserted here) -- this is what makes the honest prior-window -> future-window version of this lane worth 7 weeks instead of shipping the starter's proxy label as-is.

In [7]:
pct_declining = (df["trend_direction"] == "down").mean() * 100
pct_declining_with_demand = len(declining_with_demand) / (df["trend_direction"] == "down").sum() * 100
corr = df[["avg_position", "impressions_90d", "trend_pct"]].corr()["trend_pct"]
print(f"1) {pct_declining:.1f}% of all pages in the starter set are trending down right now.")
print(f"2) Of those declining pages, {pct_declining_with_demand:.1f}% still have real demand (impressions_90d >= 100) -- most aren't dead, they're at risk.")
print(f"3) Correlation of trend_pct with current avg_position: {corr['avg_position']:.3f}, with current impressions_90d: {corr['impressions_90d']:.3f} -- today's snapshot barely predicts tomorrow's direction, which is the whole argument for this lane.")

1) 54.2% of all pages in the starter set are trending down right now.
2) Of those declining pages, 80.9% still have real demand (impressions_90d >= 100) -- most aren't dead, they're at risk.
3) Correlation of trend_pct with current avg_position: 0.047, with current impressions_90d: 0.024 -- today's snapshot barely predicts tomorrow's direction, which is the whole argument for this lane.


## 4. Careful words: what I can and can't claim

**What this can say (once the 7 weeks are done):** an observed pattern in this dataset -- which pages, in the prior feature window, are associated with a declining or recovering trend in the following window. A directional, decision-support signal for a reviewer's queue ("these pages are more likely than average to keep declining, review them first"), with confidence bounded by the model's measured precision/recall on held-out data -- stated honestly, not implied.

**What this can never say:** that a refresh caused a recovery -- a correlation between "we refreshed X" and "X recovered" is not proof; that needs a controlled experiment with a holdout group, not a historical dataset. It also can't claim to predict or explain Google's ranking algorithm -- this only describes patterns in FlyRank's own traffic data, never search engine internals.

**A mistake I already caught:** I first thought impressions_prev_30d vs impressions_last_30d gave me a real prior-window/future-window split for free. A quick crosstab against trend_direction showed that every single page trend_direction already calls "down" also gets flagged "down" by that comparison (100% recall) -- which already proves it's not independent future evidence, it's derived from the same current-window numbers trend_pct is built from. (It isn't a perfect duplicate either: only 76% of pages the comparison flags "down" are actually trend_direction == "down", since the comparison ignores the "stable" band -- a distinction worth stating precisely instead of rounding up to "identical." See the cell below for both numbers.)

In [8]:
momentum_bucket = (df["impressions_last_30d"] > df["impressions_prev_30d"]).map({True: "up", False: "down"})
overlap = pd.crosstab(momentum_bucket, df["trend_direction"])
print(overlap)
recall_down = (momentum_bucket[df["trend_direction"] == "down"] == "down").mean() * 100
precision_down = (df["trend_direction"][momentum_bucket == "down"] == "down").mean() * 100
print(f"Of pages truly trending down, {recall_down:.0f}% also get flagged 'down' by momentum_bucket -- but of pages momentum_bucket flags 'down', only {precision_down:.0f}% are truly trend_direction == 'down'.")

trend_direction   down  flat   new  stable    up
row_0                                           
down             16262  1152     0    3896     0
up                   0     0  2236    2066  4388
Of pages truly trending down, 100% also get flagged 'down' by momentum_bucket -- but of pages momentum_bucket flags 'down', only 76% are truly trend_direction == 'down'.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.